# MemGraphRAG – Google Colab Uygulaması

**Ad Soyad / Öğrenci No:**
**Dil modeli:** Qwen3.5-9B (Ollama ile Colab GPU'sunda)
**Veri seti:** HotpotQA (repodaki örnek veri), 10 soru

## Model seçimi

Qwen3.5-9B (Alibaba, Mart 2026, Apache 2.0) modelini kullandım. Nedenleri:
- 4-bit hali yaklaşık 6.6 GB; embedding modeliyle birlikte Colab'ın T4 GPU'suna sığıyor.
- Ücretsiz, API anahtarı gerektirmiyor.
- Ollama'nın OpenAI uyumlu sunucusu sayesinde MemGraphRAG'a sadece model adı ve adres değiştirilerek bağlanabiliyor.
- MemGraphRAG her adımda JSON çıktı bekliyor; bu boyuttaki güncel modeller arasında talimat takibi en iyi olanlardan biri.

Ollama bu modelde düşünme modunu açık başlatıyor; test hücresinde kapatıyorum. Kapatılamazsa düşünme modu olmayan `ministral-3:8b` modeline geçiliyor.

## Kurulum

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /content
![ -d MemGraphRAG ] || git clone -q https://github.com/XMUDeepLIT/MemGraphRAG.git
%cd /content/MemGraphRAG

In [ ]:
# requirements.txt Colab'daki torch sürümünü değiştirdiği için sadece gerekli paketler
!pip install -q "transformers>=4.45,<5" python-igraph openai tenacity tiktoken filelock einops nest_asyncio gritlm

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, time, requests

ollama_env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="8192", OLLAMA_NUM_PARALLEL="4",
                  OLLAMA_KEEP_ALIVE="-1", OLLAMA_MAX_LOADED_MODELS="1")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=ollama_env,
                               stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print("ollama", requests.get("http://localhost:11434/api/version", timeout=2).json()["version"])
        break
    except requests.exceptions.RequestException:
        time.sleep(1)

In [ ]:
import json

LLM_NAME = "qwen3.5:9b"
LLM_BASE_URL = "http://localhost:11434/v1"
EMBEDDING_MODEL = "BAAI/bge-large-en-v1.5"

os.environ["OPENAI_API_KEY"] = "ollama"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!ollama pull {LLM_NAME} 2>&1 | tail -n 1

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=LLM_BASE_URL)
messages = [
    {"role": "system", "content": 'Extract the named entities. Respond only with JSON: {"named_entities": [...]}'},
    {"role": "user", "content": "The Newcomers is a 2000 film directed by James Allen Bradley and starring Kate Bosworth and Chris Evans."},
]
client.chat.completions.create(model=LLM_NAME, messages=messages, max_tokens=5)

LLM_EXTRA_BODY = None
for opt in [{"reasoning_effort": "none"}, {"think": False}]:
    try:
        r = client.chat.completions.create(model=LLM_NAME, messages=messages, max_tokens=300, temperature=0, extra_body=opt)
    except Exception as exc:
        print(opt, exc)
        continue
    if r.choices[0].message.content and not getattr(r.choices[0].message, "reasoning", None):
        LLM_EXTRA_BODY = opt
        break
if LLM_EXTRA_BODY is None:
    LLM_NAME, LLM_EXTRA_BODY = "ministral-3:8b", {}
    subprocess.run(["ollama", "pull", LLM_NAME], capture_output=True)
os.environ["LLM_EXTRA_BODY"] = json.dumps(LLM_EXTRA_BODY)
print("model:", LLM_NAME, "| ek parametre:", LLM_EXTRA_BODY)

start = time.time()
resp = client.chat.completions.create(model=LLM_NAME, messages=messages, max_tokens=300,
                                      temperature=0, extra_body=LLM_EXTRA_BODY or None)
elapsed = time.time() - start
print(resp.choices[0].message.content)
print(f"{resp.usage.completion_tokens / elapsed:.1f} token/sn")
!ollama ps

## Kod değişiklikleri

LLM sınıfına geçici hatalarda tekrar deneme ve modele özel parametre desteği ekledim. Ayrıca bozuk bir JSON cevabında tüm indekslemenin durmaması için `_memory_json` fonksiyonunu değiştirdim.

In [ ]:
%%writefile code/src/llm/openai_gpt.py
import functools
import re
import hashlib
import json
import os
import sqlite3
from copy import deepcopy
from typing import List, Tuple

import httpx
import openai
from filelock import FileLock
from openai import OpenAI
from packaging import version
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential

from ..utils.config_utils import BaseConfig
from ..utils.llm_utils import (
    TextChatMessage
)
from ..utils.logging_utils import get_logger
from .base import BaseLLM, LLMConfig

logger = get_logger(__name__)

def cache_response(func):
    @functools.wraps(func)
    def wrapper(self, *args, **kwargs):
        # get messages from args or kwargs
        if args:
            messages = args[0]
        else:
            messages = kwargs.get("messages")
        if messages is None:
            raise ValueError("Missing required 'messages' parameter for caching.")

        # get model, seed and temperature from kwargs or self.llm_config.generate_params
        gen_params = getattr(self, "llm_config", {}).generate_params if hasattr(self, "llm_config") else {}
        model = kwargs.get("model", gen_params.get("model"))
        seed = kwargs.get("seed", gen_params.get("seed"))
        temperature = kwargs.get("temperature", gen_params.get("temperature"))

        # build key data, convert to JSON string and hash to generate key_hash
        key_data = {
            "messages": messages,  # messages requires JSON serializable
            "model": model,
            "seed": seed,
            "temperature": temperature,
        }
        key_str = json.dumps(key_data, sort_keys=True, default=str)
        key_hash = hashlib.sha256(key_str.encode("utf-8")).hexdigest()

        # the file name of lock, ensure mutual exclusion when accessing concurrently
        lock_file = self.cache_file_name + ".lock"

        # Try to read from SQLite cache
        with FileLock(lock_file):
            conn = sqlite3.connect(self.cache_file_name)
            c = conn.cursor()
            # if the table does not exist, create it
            c.execute("""
                CREATE TABLE IF NOT EXISTS cache (
                    key TEXT PRIMARY KEY,
                    message TEXT,
                    metadata TEXT
                )
            """)
            conn.commit()  # commit to save the table creation
            c.execute("SELECT message, metadata FROM cache WHERE key = ?", (key_hash,))
            row = c.fetchone()
            conn.close()
            if row is not None:
                message, metadata_str = row
                metadata = json.loads(metadata_str)
                # return cached result and mark as hit
                return message, metadata, True

        # if cache miss, call the original function to get the result
        result = func(self, *args, **kwargs)
        message, metadata = result

        # insert new result into cache
        with FileLock(lock_file):
            conn = sqlite3.connect(self.cache_file_name)
            c = conn.cursor()
            # make sure the table exists again (if it doesn't exist, it would be created)
            c.execute("""
                CREATE TABLE IF NOT EXISTS cache (
                    key TEXT PRIMARY KEY,
                    message TEXT,
                    metadata TEXT
                )
            """)
            metadata_str = json.dumps(metadata)
            c.execute("INSERT OR REPLACE INTO cache (key, message, metadata) VALUES (?, ?, ?)",
                      (key_hash, message, metadata_str))
            conn.commit()
            conn.close()

        return message, metadata, False

    return wrapper


RETRYABLE_ERRORS = (openai.RateLimitError, openai.APIConnectionError,
                    openai.APITimeoutError, openai.InternalServerError)

THINK_PATTERN = re.compile(r"<think>.*?</think>", re.DOTALL)


class CacheOpenAI(BaseLLM):
    """OpenAI LLM implementation."""
    @classmethod
    def from_experiment_config(cls, global_config: BaseConfig) -> "CacheOpenAI":
        config_dict = global_config.__dict__
        cache_dir = os.path.join(global_config.save_dir, "llm_cache")
        return cls(cache_dir=cache_dir, **config_dict)

    def __init__(self, cache_dir, cache_filename: str = None,
                 llm_name: str = "gpt-4o-mini", api_key: str = None, llm_base_url: str = None, 
                 high_throughput: bool = True,
                 **kwargs) -> None:
        super().__init__()
        self.cache_dir = cache_dir
        os.makedirs(self.cache_dir, exist_ok=True)
        if cache_filename is None:
            cache_filename = f"{llm_name.replace('/', '_')}_cache.sqlite"
        self.cache_file_name = os.path.join(self.cache_dir, cache_filename)
        self.llm_name = llm_name
        self.llm_base_url = llm_base_url
        self.extra_body = json.loads(os.getenv("LLM_EXTRA_BODY", "{}") or "{}")
        self._init_llm_config(**kwargs)
        if high_throughput:
            limits = httpx.Limits(max_connections=500, max_keepalive_connections=100)
            client = httpx.Client(limits=limits, timeout=httpx.Timeout(5*60, read=5*60))
        else:
            client = None
        self.openai_client = OpenAI(base_url=self.llm_base_url, api_key=api_key, http_client=client)

    def _init_llm_config(self, **kwargs) -> None:
        config_dict = {
            "llm_name": self.llm_name,
            "llm_base_url": self.llm_base_url,
            "generate_params": {
                "model": self.llm_name,
                "max_completion_tokens": kwargs.get("max_new_tokens", 400),
                "n": kwargs.get("num_gen_choices", 1),
                "seed": kwargs.get("seed", 0),
                "temperature": kwargs.get("temperature", 0.0),
            }
        }
        self.llm_config = LLMConfig.from_dict(config_dict=config_dict)
        logger.debug(f"Init {self.__class__.__name__}'s llm_config: {self.llm_config}")

    @cache_response
    def infer(
        self,
        messages: List[TextChatMessage],
        **kwargs
    ) -> Tuple[List[TextChatMessage], dict]:
        params = deepcopy(self.llm_config.generate_params)
        if kwargs:
            params.update(kwargs)
        params["messages"] = messages
        logger.debug(f"Calling OpenAI GPT API with:\n{params}")

        if 'gpt' not in params['model'] or version.parse(openai.__version__) < version.parse("1.45.0"): # if we use vllm to call openai api or if we use openai but the version is too old to use 'max_completion_tokens' argument
            # TODO strange version change in openai protocol, but our current vllm version not changed yet
            params['max_tokens'] = params.pop('max_completion_tokens')

        if self.extra_body:
            params["extra_body"] = self.extra_body

        response = self._create_with_retry(params)

        response_message = response.choices[0].message.content or ""
        response_message = THINK_PATTERN.sub("", response_message).strip()

        metadata = {
            "prompt_tokens": response.usage.prompt_tokens, 
            "completion_tokens": response.usage.completion_tokens,
            "finish_reason": response.choices[0].finish_reason,
        }

        return response_message, metadata

    @retry(retry=retry_if_exception_type(RETRYABLE_ERRORS), stop=stop_after_attempt(6),
           wait=wait_exponential(multiplier=2, min=2, max=60), reraise=True)
    def _create_with_retry(self, params):
        return self.openai_client.chat.completions.create(**params)

In [ ]:
from pathlib import Path

path = Path("code/src/MemGraphRAG.py")
src = path.read_text()
old = "            return json.loads(fix_broken_generated_json(candidate))\n"
new = ("            try:\n"
       "                return json.loads(fix_broken_generated_json(candidate))\n"
       "            except Exception:\n"
       "                logger.warning(\"Memory JSON could not be parsed, skipping: %s\", cleaned[:200])\n"
       "                return {}\n")
if new not in src:
    path.write_text(src.replace(old, new))

## Veri hazırlama

HotpotQA'dan ilk 10 soruyu aldım. Her soru için destek paragraflarını ve 2 ilgisiz paragrafı korpusa ekledim.

In [ ]:
import json
import os
import random

N_QUESTIONS = 10
N_DISTRACTORS = 2
DATA_DIR = "data/hotpotqa_small"

random.seed(42)
with open("dataset/hotpotqa/hotpotqa.json", encoding="utf-8") as f:
    hotpot = json.load(f)

selected = hotpot[:N_QUESTIONS]
paragraphs, seen_titles = [], set()
for sample in selected:
    support_titles = {title for title, _ in sample["supporting_facts"]}
    context = {title: sentences for title, sentences in sample["context"]}
    distractors = [t for t in context if t not in support_titles]
    chosen = sorted(support_titles) + random.sample(distractors, min(N_DISTRACTORS, len(distractors)))
    for title in chosen:
        if title in seen_titles or title not in context:
            continue
        seen_titles.add(title)
        paragraphs.append(f"{title}\n{''.join(context[title]).strip()}")

random.shuffle(paragraphs)
questions = [
    {"_id": s["_id"], "question": s["question"].strip(), "answer": s["answer"],
     "type": s["type"], "supporting_facts": s["supporting_facts"]}
    for s in selected
]

os.makedirs(DATA_DIR, exist_ok=True)
with open(f"{DATA_DIR}/corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(paragraphs))
with open(f"{DATA_DIR}/questions.json", "w", encoding="utf-8") as f:
    json.dump(questions, f, ensure_ascii=False, indent=2)

print("paragraf:", len(paragraphs), "| soru:", len(questions))

## İndeksleme

In [ ]:
SAVE_DIR = "outputs/hotpotqa_small"
index_start = time.time()
!python code/index.py \
  --corpus data/hotpotqa_small/corpus.txt \
  --save-dir {SAVE_DIR} \
  --llm-name {LLM_NAME} \
  --llm-base-url {LLM_BASE_URL} \
  --embedding-model {EMBEDDING_MODEL} \
  --chunk-size 256 --chunk-overlap 32 \
  --memory-max-workers 4 \
  --conflict-max-related 5 \
  --artifact-mode detailed 2>&1 | grep -vE "httpx|it/s\]|s/it\]|^\s*$"
print(f"süre: {(time.time() - index_start) / 60:.1f} dk")

## Bellek ve graf

In [ ]:
import pandas as pd

load = lambda name: json.load(open(f"{SAVE_DIR}/{name}", encoding="utf-8"))
with_schema = load("initial_memory_with_schema.json")
filtered = load("memory_after_ontology_filtering.json")
final = load("memory.json")
conflict = load("conflict_detection_results.json")
resolve = load("conflict_resolution_results.json")
mgraph = load("graph_from_memory/memory_graph.json")

sizes = lambda m: {k: len(m.get(k, [])) for k in ("schema_layer", "fact_layer", "passage_layer")}
display(pd.DataFrame({"ontoloji": sizes(with_schema), "filtre": sizes(filtered), "çelişki çözümü": sizes(final)}))
print("çelişki:", conflict["summary"]["num_conflict_items"], "| kararlar:", resolve["summary"]["actions_applied"])
print("graf:", mgraph["stats"])

In [ ]:
import re
import networkx as nx
import matplotlib.pyplot as plt

norm_text = lambda s: re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()
nodes_by_id = {n["id"]: n for n in mgraph["nodes"]}
adj = {}
for e in mgraph["edges"]:
    adj.setdefault(e["source"], []).append(e)

questions = json.load(open("data/hotpotqa_small/questions.json", encoding="utf-8"))
title = norm_text(questions[0]["supporting_facts"][0][0])
passages_with_title = [n["id"] for n in mgraph["nodes"] if n["layer"] == "passage" and title in norm_text(n["content"])]
passage_id = max(passages_with_title or [n["id"] for n in mgraph["nodes"] if n["layer"] == "passage"],
                 key=lambda pid: sum(e["type"] == "passage_entity" for e in adj.get(pid, [])))

G = nx.Graph()
G.add_node(passage_id, layer="passage", label="pasaj\n" + nodes_by_id[passage_id]["content"].split(":", 1)[-1][:30] + "...")
entities = [e["target"] for e in adj.get(passage_id, []) if e["type"] == "passage_entity"][:12]
for ent in entities:
    G.add_node(ent, layer="entity", label=nodes_by_id[ent]["label"][:25])
    G.add_edge(passage_id, ent)
    related = [e for e in adj.get(ent, []) if e["type"] == "entity_relation" and e["target"] not in entities][:2]
    for e in related + [e for e in adj.get(ent, []) if e["type"] == "entity_to_type"]:
        G.add_node(e["target"], layer=nodes_by_id[e["target"]]["layer"], label=nodes_by_id[e["target"]]["label"][:25])
        G.add_edge(ent, e["target"])

colors = {"passage": "#eb6834", "entity": "#2a78d6", "type": "#1baf7a"}
plt.figure(figsize=(13, 9))
pos = nx.spring_layout(G, seed=3, k=1.2)
for layer, color in colors.items():
    nodelist = [n for n in G if G.nodes[n]["layer"] == layer]
    nx.draw_networkx_nodes(G, pos, nodelist=nodelist, node_color=color, label=layer,
                           node_size=900 if layer == "passage" else 380, node_shape="s" if layer == "type" else "o")
nx.draw_networkx_edges(G, pos, alpha=0.35)
nx.draw_networkx_labels(G, pos, nx.get_node_attributes(G, "label"), font_size=8)
plt.title(questions[0]["supporting_facts"][0][0])
plt.legend(frameon=False, markerscale=0.6); plt.axis("off"); plt.show()

## Soru-cevap

In [ ]:
!python code/retrieval_dataset_test.py \
  --questions data/hotpotqa_small/questions.json \
  --save-dir {SAVE_DIR} \
  --output results/hotpotqa_small/memgraphrag_qa.json \
  --llm-name {LLM_NAME} \
  --llm-base-url {LLM_BASE_URL} \
  --embedding-model {EMBEDDING_MODEL} \
  --sample-num 0 --qa-top-k 5 --qa-max-workers 4 \
  --skip-fact-rerank true \
  --fact-similarity-threshold 0.4 \
  --use-raw-threshold-filter true 2>&1 | grep -vE "httpx|it/s\]|s/it\]|^\s*$"

In [ ]:
import string
from collections import Counter

def normalize(s):
    s = "".join(ch for ch in str(s).lower() if ch not in set(string.punctuation))
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", s).split())

def em(pred, gold):
    return float(normalize(pred) == normalize(gold))

def f1(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0:
        return 0.0
    return 2 * common / (len(p) + len(g))

def support_recall(docs, supporting_facts, k=5):
    simplify = lambda s: " " + re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip() + " "
    titles = {t for t, _ in supporting_facts}
    text = simplify(" ".join(docs[:k]))
    return sum(simplify(t) in text for t in titles) / len(titles)

mem = json.load(open("results/hotpotqa_small/memgraphrag_qa.json", encoding="utf-8"))
mem_df = pd.DataFrame([{
    "soru": r["question"], "doğru cevap": r["answer"], "cevap": r["predicted_answer"],
    "EM": em(r["predicted_answer"], r["answer"]), "F1": round(f1(r["predicted_answer"], r["answer"]), 2),
    "Recall@5": support_recall(r["retrieved_docs"], r["supporting_facts"]),
} for r in mem["results"]])
pd.set_option("display.max_colwidth", 80)
display(mem_df)

## Klasik RAG ile karşılaştırma

Aynı korpus, embedding modeli ve LLM ile, graf kullanmadan sadece vektör benzerliğiyle en yakın 5 parçayı getirdim.

In [ ]:
import sys, logging
sys.path.insert(0, "/content/MemGraphRAG/code")
logging.getLogger().setLevel(logging.WARNING)
from src.MemGraphRAG import MemGraphRAG
from src.utils.config_utils import BaseConfig
from src.utils.misc_utils import QuerySolution

config = BaseConfig(save_dir=SAVE_DIR, llm_name=LLM_NAME, llm_base_url=LLM_BASE_URL,
                    embedding_model_name=EMBEDDING_MODEL, openie_mode="online",
                    force_index_from_scratch=False, qa_top_k=5, qa_max_workers=4, max_new_tokens=1000)
rag = MemGraphRAG(global_config=config)
rag.prepare_retrieval_objects()

qs = [q["question"] for q in questions]
rag.get_query_embeddings(qs)
dense_solutions = []
for q in qs:
    ids, scores = rag.dense_passage_retrieval(q)
    docs = [rag.chunk_embedding_store.get_row(rag.passage_node_keys[i])["content"] for i in ids[:5]]
    dense_solutions.append(QuerySolution(question=q, docs=docs, doc_scores=scores[:5]))
dense_solutions, _, _ = rag.qa(dense_solutions)

dense_df = pd.DataFrame([{
    "soru": q["question"], "doğru cevap": q["answer"], "cevap": s.answer,
    "EM": em(s.answer, q["answer"]), "F1": round(f1(s.answer, q["answer"]), 2),
    "Recall@5": support_recall(s.docs, q["supporting_facts"]),
} for q, s in zip(questions, dense_solutions)])
display(dense_df)

In [ ]:
import numpy as np

metrics = ["EM", "F1", "Recall@5"]
summary = pd.DataFrame({"Klasik RAG": dense_df[metrics].mean(), "MemGraphRAG": mem_df[metrics].mean()}).T.round(3)
display(summary)

x = np.arange(len(metrics)); w = 0.36
fig, ax = plt.subplots(figsize=(7, 4))
for i, (name, color) in enumerate([("Klasik RAG", "#eb6834"), ("MemGraphRAG", "#2a78d6")]):
    bars = ax.bar(x + (i - 0.5) * w, summary.loc[name, metrics], w - 0.02, label=name, color=color)
    ax.bar_label(bars, fmt="%.2f", fontsize=9, color="#52514e", padding=2)
ax.set_xticks(x, metrics); ax.set_ylim(0, 1.1)
ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.25); ax.set_axisbelow(True)
ax.legend(frameon=False); plt.tight_layout(); plt.show()

## Karşılaşılan hatalar

| Hata | Neden | Çözüm |
|---|---|---|
| `requirements.txt` kurulumu Colab'daki torch'u 2.5.1'e düşürmeye çalışıyor, vllm kurulumu çok uzun sürüyor | Sürümler yazarların ortamına göre sabitlenmiş, vllm sadece offline mod için gerekli | Sadece gerekli paketleri kurdum |
| `ModuleNotFoundError: No module named 'gritlm'` | GritLM modülü kullanılmasa da koşulsuz import ediliyor | `gritlm` paketini kurdum |
| README'deki `datasets/corpus/corpus.txt` bulunamadı | Klasörün adı `dataset/`, örnek dosyalar repoda yok | `dataset/hotpotqa` verisinden kendi korpusumu oluşturdum |
| `run_index.sh`: `PYTHON: unbound variable` | Betikte tanımsız `$PYTHON` değişkeni ve yazarın kendi dosya yolları var | `index.py`'yi doğrudan çalıştırdım |
| İndekslemede her istek ~80 sn sürdü, `Retrying request` hataları geldi; testte modelin cevabı boş döndü | Ollama, Qwen3.5-9B'de düşünme modunu açık başlatıyor; model her cevaptan önce yüzlerce token düşünce üretiyor. Ollama bu modelde istekleri tek tek işlediği için sıradaki istekler zaman aşımına uğruyor | İsteklere düşünme modunu kapatan parametreyi ekledim (`reasoning_effort: "none"`) |

## Değerlendirme

Klasik RAG soruya en çok benzeyen parçaları getiriyor. Çok adımlı sorularda ikinci adımdaki paragraf soruya benzemediği için bulunamayabiliyor. GraphRAG metinden varlık-ilişki grafı çıkarıp graf üzerinde dolaşarak bu bağlantıyı kurabiliyor, fakat LLM'in çıkardığı hatalı veya çelişkili bilgiler de grafa giriyor.

MemGraphRAG grafı kurmadan önce bilgiyi şema, olgu ve pasaj katmanlarına ayırıyor. Nadir şemaları eliyor ve çelişen olguları kaynak pasaja bakarak düzeltiyor; graf bu temizlenmiş bellekten oluşturuluyor. Bunun bedeli indekslemedeki çok sayıda LLM çağrısı: klasik RAG indekslemede hiç LLM kullanmıyor.

_(Sonuçlar çalıştırmadan sonra eklenecek.)_

10 soru küçük bir örneklem olduğu için sonuçlar kesin bir karşılaştırma için yeterli değil.